# Notebook 10 — Cell Signaling and Gene Regulation

**Module:** 05 — Biology Fundamentals  
**Notebook:** 10 of 18  
**Prerequisites:** NB03  
**Time estimate:** 75 minutes

---
## Step 1 — Motivation

ChIP-seq, ATAC-seq, Hi-C, and single-cell multiomics all measure aspects of gene
regulation. Transcription factor binding, chromatin accessibility, DNA looping — these
are the regulatory machinery. Without understanding *why* a gene's expression changes
in response to a signal, you can't interpret what a regulatory genomics dataset means.

---
## Step 2 — Intuition

A cell is a signal-processing machine. Signals arrive at the cell surface (hormones,
growth factors, neurotransmitters), are detected by receptors, amplified through a
cascade of protein modifications, and ultimately change which genes are turned on or
off. Think of it as an event-driven system: `signal_received()` triggers a chain of
function calls that ends with `write_gene_expression_file()`.

---
## Step 3 — Biological Background

**Core signaling components:**
1. **Ligand:** the signal molecule (e.g., EGF, insulin, Wnt)
2. **Receptor:** cell surface or intracellular protein that detects the ligand
3. **Signal cascade:** chain of protein activations (often by phosphorylation)
4. **Transcription factor (TF):** the final step; enters nucleus, binds DNA, activates/represses genes

**MAPK/ERK cascade (prototype example):**
```
EGF → EGFR → Ras → Raf → MEK → ERK → transcription factors (e.g. c-Fos)
```
Each arrow typically involves phosphorylation — adding a phosphate group to a protein,
changing its shape and activity.

**Gene regulation at the DNA level:**
- **Transcription factors:** bind specific DNA sequences (motifs) near gene promoters
  or at enhancers; recruit RNA polymerase or repressors
- **Chromatin remodeling:** histones can be modified (acetylation opens chromatin;
  methylation generally closes it) — this is what ATAC-seq and ChIP-seq measure
- **Enhancers:** can be far (Mb) from their target gene; brought close by DNA looping
  (measured by Hi-C)

**Boolean gene regulatory network (simplified):**
Each gene is ON or OFF; its state in the next time step is a Boolean function of
the current states of its regulators. Used to model development and cell fate decisions.

**Feedback loops:**
- **Negative feedback:** output inhibits its own production → stabilises the system
  (e.g., p53 activates Mdm2, which degrades p53)
- **Positive feedback:** output promotes its own production → bistability, switch-like
  behaviour (e.g., cell fate commitment)

---
## Step 4 — Mathematical Explanation

**ODE model of a simple negative feedback loop:**

$$\frac{d[X]}{dt} = \alpha - \beta [X] - \gamma [X][Y]$$
$$\frac{d[Y]}{dt} = \delta [X] - \varepsilon [Y]$$

where X is a transcription factor and Y is its repressor.

**Michaelis-Menten kinetics for TF binding:**
$$v = V_{max} \frac{[S]}{K_m + [S]}$$

At high [S]: v → Vmax (saturated). At [S] = Km: v = Vmax/2.
This describes how transcription rate saturates as TF concentration increases.

---
## Step 6 — Python Implementation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

In [ ]:
# Cell 6.1 — ODE model: MAPK cascade (simplified Raf-MEK-ERK)
def mapk_cascade(t, y, k_on, k_off):
    """
    Simplified 3-stage activation cascade.
    y = [Raf_active, MEK_active, ERK_active]
    Activation proportional to upstream active concentration;
    deactivation at constant rate.
    """
    raf, mek, erk = y

    # Input signal: step function at t=5 min
    signal = 1.0 if t >= 5.0 else 0.0

    d_raf = k_on[0] * signal * (1 - raf) - k_off[0] * raf
    d_mek = k_on[1] * raf   * (1 - mek) - k_off[1] * mek
    d_erk = k_on[2] * mek   * (1 - erk) - k_off[2] * erk

    return [d_raf, d_mek, d_erk]

k_on  = [0.5, 0.8, 0.6]   # activation rate constants
k_off = [0.1, 0.1, 0.1]   # deactivation rates

sol = solve_ivp(
    mapk_cascade, [0, 40], [0, 0, 0],
    args=(k_on, k_off), dense_output=True, max_step=0.1
)

t_eval = np.linspace(0, 40, 400)
y_eval = sol.sol(t_eval)

print(f"Signal arrives at t=5 min")
print(f"ERK peak activation: {y_eval[2].max():.3f} (fraction active)")
print(f"ERK half-max time:   ~{t_eval[np.argmax(y_eval[2] > 0.5)]:.1f} min")

In [ ]:
# Cell 6.2 — Boolean gene regulatory network
def boolean_grn_step(state: dict, rules: dict) -> dict:
    """
    One synchronous update step of a Boolean gene regulatory network.
    state: dict {gene_name: 0/1}
    rules: dict {gene_name: function(state) -> 0/1}
    """
    return {gene: int(rules[gene](state)) for gene in state}

# Simple 3-gene network: A activates B; B activates C; C represses A
rules = {
    'A': lambda s: 1 if (s['A'] == 1 and s['C'] == 0) else (1 if s['A'] == 0 and s['C'] == 0 else 0),
    'B': lambda s: int(s['A']),
    'C': lambda s: int(s['B']),
}

# Run from two different initial conditions
print("Boolean GRN: A activates B, B activates C, C represses A")
for init in [{'A': 1, 'B': 0, 'C': 0}, {'A': 0, 'B': 0, 'C': 0}]:
    state = init.copy()
    trajectory = [state.copy()]
    for _ in range(8):
        state = boolean_grn_step(state, rules)
        trajectory.append(state.copy())
    print(f"  Init {init}: ", end='')
    for s in trajectory:
        print(f"({'A' if s['A'] else '·'}{'B' if s['B'] else '·'}{'C' if s['C'] else '·'})", end=' ')
    print()

---
## Step 7 — Visualization

In [ ]:
# Cell 7.1 — MAPK cascade dynamics
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Panel 1: Cascade activation over time
ax = axes[0]
labels = ['Raf (layer 1)', 'MEK (layer 2)', 'ERK (layer 3)']
colors = ['steelblue', 'tomato', 'seagreen']
for i, (label, color) in enumerate(zip(labels, colors)):
    ax.plot(t_eval, y_eval[i], label=label, color=color, lw=2)
ax.axvline(5, color='gray', ls='--', lw=1, label='Signal (t=5 min)')
ax.set_xlabel('Time (min)')
ax.set_ylabel('Fraction active (0–1)')
ax.set_title('MAPK/ERK cascade: sequential activation\nSignal propagates with delay and amplification')
ax.legend(fontsize=9)

# Panel 2: Signal delay at each cascade layer
ax = axes[1]
half_max_times = []
for i in range(3):
    idx = np.where(y_eval[i] > 0.5)[0]
    half_max_times.append(t_eval[idx[0]] if len(idx) > 0 else np.nan)

ax.bar(['Raf', 'MEK', 'ERK'], [t - 5 for t in half_max_times],
       color=colors, alpha=0.8, edgecolor='gray')
ax.set_ylabel('Delay after signal (min)')
ax.set_title('Cascade introduces signal delay at each layer\n(amplification comes with latency)')

plt.tight_layout()
plt.show()

---
## Step 8 — Exercises

1. Modify the MAPK cascade ODE to include a negative feedback: ERK phosphorylates
   and inactivates Raf (add a term `−k_fb * erk * raf` to d_raf). How does
   the temporal profile of ERK change?
2. Draw the regulatory network for p53/Mdm2 negative feedback:
   - p53 activates Mdm2 expression
   - Mdm2 targets p53 for degradation
   What is the steady state? What happens if Mdm2 is deleted?
3. ChIP-seq for transcription factor X shows peaks near 200 genes in cancer cells
   but not in normal cells. What does this tell you about the signaling cascade
   that activates X? What experiment would you run next?
4. What is the difference between an activator TF and a repressor TF? How would
   you detect each using ChIP-seq combined with RNA-seq data?

---
## Quiz — Active Recall

1. Name the four components of a signaling pathway (from signal to gene).
2. What is phosphorylation and why is it used in signaling cascades?
3. What is a transcription factor? How does it control gene expression?
4. What is the difference between an enhancer and a promoter?
5. What does ATAC-seq measure? How does it relate to gene regulation?

---
## Reflection

**Date completed:** ____________________

> *[A gene is highly expressed in liver but not in brain. Name two molecular mechanisms
> that could explain tissue-specific expression. Which would you look for first,
> and with what assay?]*

---
*Next: `11_ecology_population_biology.ipynb`*